# 07 — Customer lifetime value, and a model that does not fit

3.03% of Olist's customers ever bought twice, and those who did averaged 1.11
repeat purchases. This notebook establishes what follows from that, which is more
interesting than a fitted curve would have been.

In [1]:
import json
import warnings

import pandas as pd

from athar import paths
from athar.provenance import read_metric

warnings.filterwarnings("ignore")
pd.set_option("display.width", 200)
pd.set_option("display.max_columns", 50)

METRICS = paths.metrics_dir()
PROCESSED = paths.processed_dir()


def show(frame, caption=""):
    if caption:
        print(caption)
    print(frame.to_string(index=False))
    print()

In [2]:
try:
    clv = read_metric("clv", METRICS)
except FileNotFoundError:
    clv = None
    print("metrics/clv.json not present; run `make clv`.")

if clv:
    print(json.dumps(clv["repeat_behaviour"], indent=2))

{
  "customers": 93573,
  "max_repeats": 14,
  "mean_repeats_among_repeaters": 1.112755461592671,
  "mean_repeats_overall": 0.03374905154264585,
  "repeat_rate": 0.030329261645987624,
  "repeaters": 2838,
  "with_two_or_more_repeats": 230,
  "zero_repeat_share": 0.9696707383540124
}


## Maximum likelihood does not converge, and it is not a tuning problem

BG/NBD is the standard model and `lifetimes` is the reference implementation. It
does not converge here — at any time scale, at any penalty, on the full base or on
the repeaters alone. The likelihood returns NaN and the parameters run off in log
space.

The reason is structural. BG/NBD's dropout parameters describe the shape of a Beta
distribution over the probability of churning after each purchase, and they are
identified only by the *pattern* of repeat purchasing. With repeaters averaging
1.11 repeats there is no pattern for them to be estimated from, so the likelihood
is flat in those directions.

In [3]:
if clv:
    ml = clv["maximum_likelihood"]
    show(pd.DataFrame(ml["attempts_full_base"])[["time_unit", "penalizer", "converged"]],
         "Full base")
    show(pd.DataFrame(ml["attempts_repeaters_only"])[["time_unit", "penalizer", "converged"]],
         "Repeaters only")
    print(ml["finding"])

Full base
time_unit  penalizer  converged
     days       0.00      False
     days       0.01      False
     days       0.10      False
     days       1.00      False
     days      10.00      False
    weeks       0.00      False
    weeks       0.01      False
    weeks       0.10      False
    weeks       1.00      False
    weeks      10.00      False
   months       0.00      False
   months       0.01      False
   months       0.10      False
   months       1.00      False
   months      10.00      False

Repeaters only
time_unit  penalizer  converged
     days       0.00      False
     days       0.01      False
     days       0.10      False
     days       1.00      False
     days      10.00      False
    weeks       0.00      False
    weeks       0.01      False
    weeks       0.10      False
    weeks       1.00      False
    weeks      10.00      False
   months       0.00      False
   months       0.01      False
   months       0.10      False
   months     

## The Bayesian fit runs, and should not be believed either

The MCMC fit is the only one that runs at all, and running is not the same as
working. Two things have to be checked before a posterior is quoted, and both fail
here.

The sampler's own diagnostics: the calibration fit throws thousands of divergences.

And degeneracy. A narrow posterior is not the same as an informed one — `alpha`
lands near 1e-306, the smallest number a float can hold, against a prior mean near
9. An interval that tight is the sampler falling into a corner, not the data
speaking, and reading it as a confident estimate would be exactly the error this
project exists to avoid.

In [4]:
if clv:
    parameters = clv["models"]["bgnbd_bayesian_full_base"]["parameters"]
    print("declared priors:")
    for name, prior in parameters.get("declared_priors", {}).items():
        print(f"  {name:16s} {prior}")
    print()
    print("sampler:", parameters.get("sampler"))
    print("degenerate:", clv["models"]["bgnbd_bayesian_full_base"].get("degenerate_parameters"))
    print("trustworthy:", clv["models"]["bgnbd_bayesian_full_base"].get("trustworthy"))
    print()
    rows = [{"parameter": name, "posterior_mean": entry["posterior_mean"],
             "posterior_sd": entry["posterior_sd"], "prior_mean": entry.get("prior_mean"),
             "degenerate": entry.get("degenerate")}
            for name, entry in parameters.items()
            if isinstance(entry, dict) and "posterior_mean" in entry]
    if rows:
        show(pd.DataFrame(rows))
    print(clv["models"]["bgnbd_bayesian_full_base"]["note"])

declared priors:
  alpha            Prior("Weibull", alpha=2, beta=10)
  kappa_dropout    Prior("Pareto", alpha=1, m=1)
  phi_dropout      Prior("Uniform", lower=0, upper=1)
  r                Prior("Weibull", alpha=2, beta=1)

sampler: {'divergences': 3507, 'max_r_hat': 1.135311, 'min_ess_bulk': 20.23, 'passed': False}
degenerate: ['alpha']
trustworthy: False

parameter  posterior_mean  posterior_sd  prior_mean  degenerate
        a    2.016013e+00      0.588582    3.448737       False
    alpha   1.602094e-306      0.000000    8.918931        True
        b    2.315003e-01      0.075801    4.369117       False
        r    4.451746e-05      0.000001    0.870528       False

The MCMC fit is not a second opinion here; it is the only fit that runs at all. It should not be read as a working model. The sampler block records its divergences, and any parameter that collapsed to a point is named in degenerate_parameters — on this base `alpha` lands near 1e-306, which is the smallest number t

## Validation on a time-based holdout

Never a random split: the prediction is "how many purchases in the next N weeks",
and a random split would let the model see the future of the customers it is
forecasting.

The comparison that matters is against a baseline that predicts nothing. A
near-zero prediction matching a near-zero outcome is the model working; it is only
*skill* if it beats predicting zero.

In [5]:
if clv:
    validation = {k: v for k, v in clv["validation"].items() if not isinstance(v, str)}
    for key, value in validation.items():
        print(f"{key:44s} {value}")
    print()
    print(clv["validation"]["metric_note"])

actual_mean                                  0.007987266261328438
actual_total                                 557.0
beats_predicting_zero                        False
customers                                    69736
customers_with_history                       69736
holdout_weeks                                16
horizon_weeks                                16.0
mean_absolute_error                          0.00852733956190811
mean_absolute_error_predicting_zero          0.007987266261328438
predicted_mean                               0.0006256720280442931
predicted_total                              43.63186454769682
share_of_customers_predicted_below_0_1       0.9991826316393254

Mean absolute error, not MAPE: the actual holdout count is zero for almost every customer and a percentage error against zero is undefined. No MAPE is computed anywhere in this project.


## What lifetime value comes to, and why it decides notebook 08

If lifetime value is very nearly proportional to first-order value, then weighting
a media allocation by lifetime value and weighting it by immediate revenue rank the
channels identically and produce the same budget. The CLV-weighted reallocation the
brief invites is then not a different answer — it is the same answer with more
steps, and saying so is the finding.

In [6]:
if clv:
    value = clv["lifetime_value"]
    for key, item in value.items():
        print(f"{key:44s} {item}")
    print()
    print("VERDICT:", clv.get("verdict", ""))
    print()
    print(clv["finding"])

clv_over_first_order_value                   None
computable                                   False
correlation_clv_with_first_order_value       None
horizon_weeks                                52.0
mean_expected_clv_brl                        None
mean_first_order_value_brl                   138.11620381947782
mean_predicted_purchases                     None
median_predicted_purchases                   None
share_predicted_below_0_1_purchases          None
why_not                                      The BG/NBD posterior is degenerate — alpha sits at roughly 1e-306 — so the expected-purchases calculation divides by a denormal and returns NaN for every customer. There is no lifetime value to report because there is no working transaction model to compute it from. That is the finding, not a gap in it.

VERDICT: BG/NBD cannot be fitted to this base by any method attempted. Maximum likelihood does not converge at any setting. MCMC runs, but drives alpha to a denormal value and throws t